In [ ]:
!pip install -q gradio plotly scikit-learn reportlab requests beautifulsoup4 sentence-transformers faiss-cpu openai google-genai anthropic

In [ ]:
import getpass
import os

API_KEY = getpass.getpass("Enter your API key: ")

if API_KEY.strip():
    os.environ["API_KEY"] = API_KEY.strip()
    print("API key entered successfully!")
else:
    raise ValueError("No API key entered. Please run this cell again.")

Enter your API key: ··········
API key entered successfully!


In [ ]:
API_KEY = os.environ.get("API_KEY", "").strip()

if not API_KEY:
    raise ValueError("API key is missing. Run Cell 2 first.")

if API_KEY.startswith("sk-ant-"):
    API_PROVIDER = "anthropic"

elif API_KEY.startswith("gsk_"):
    API_PROVIDER = "groq"

elif API_KEY.startswith("AIza"):
    API_PROVIDER = "gemini"

elif API_KEY.startswith("sk-"):
    API_PROVIDER = "openai"

else:
    API_PROVIDER = "unknown"

print("Detected API provider:", API_PROVIDER)

if API_PROVIDER == "unknown":
    raise ValueError(
        "The provider could not be identified from this key prefix. "
        "Check which company issued your API key before continuing."
    )

Detected API provider: openai


In [ ]:
import os
import tempfile
import warnings
import requests
import numpy as np
import pandas as pd
import gradio as gr
import plotly.express as px
import plotly.graph_objects as go

from bs4 import BeautifulSoup

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

from sentence_transformers import SentenceTransformer
import faiss

from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas

warnings.filterwarnings("ignore")

print("All libraries imported successfully.")

All libraries imported successfully.


In [ ]:
from openai import OpenAI
from google import genai
from anthropic import Anthropic

API_KEY = os.environ.get("API_KEY", "").strip()

if API_PROVIDER == "openai":
    llm_client = OpenAI(api_key=API_KEY)
    LLM_MODEL = "gpt-4o-mini"

elif API_PROVIDER == "groq":
    llm_client = OpenAI(
        api_key=API_KEY,
        base_url="https://api.groq.com/openai/v1"
    )
    LLM_MODEL = "llama-3.3-70b-versatile"

elif API_PROVIDER == "gemini":
    llm_client = genai.Client(api_key=API_KEY)
    LLM_MODEL = "gemini-2.0-flash"

elif API_PROVIDER == "anthropic":
    llm_client = Anthropic(api_key=API_KEY)
    LLM_MODEL = "claude-3-5-haiku-latest"

else:
    raise ValueError("Unsupported API provider.")

print("API integration configured for:", API_PROVIDER)

API integration configured for: openai


In [ ]:
diabetes_url = (
    "https://raw.githubusercontent.com/plotly/datasets/"
    "master/diabetes.csv"
)

diabetes_df = pd.read_csv(diabetes_url)

diabetes_features = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age"
]

diabetes_target = "Outcome"

diabetes_df = diabetes_df[
    diabetes_features + [diabetes_target]
].copy()

print("Diabetes dataset loaded successfully.")
print("Dataset shape:", diabetes_df.shape)
display(diabetes_df.head())

Diabetes dataset loaded successfully.
Dataset shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [ ]:
heart_urls = [
    "https://raw.githubusercontent.com/sharmaroshan/"
    "Heart-UCI-Dataset/master/heart.csv",

    "https://raw.githubusercontent.com/kb22/"
    "Heart-Disease-Prediction/master/dataset.csv"
]

heart_df = None

for url in heart_urls:
    try:
        temp_df = pd.read_csv(url)

        temp_df.columns = [
            str(col).strip().lower().replace(" ", "_")
            for col in temp_df.columns
        ]

        target_candidates = [
            "target", "num", "condition",
            "output", "heartdisease"
        ]

        target_col = next(
            (
                col for col in target_candidates
                if col in temp_df.columns
            ),
            None
        )

        if target_col is None:
            continue

        heart_df = temp_df.rename(
            columns={target_col: "target"}
        ).copy()

        heart_df["target"] = pd.to_numeric(
            heart_df["target"], errors="coerce"
        )

        heart_df = heart_df.dropna(subset=["target"])

        heart_df["target"] = (
            heart_df["target"] > 0
        ).astype(int)

        break

    except Exception as error:
        print("Trying the next heart dataset source.")

if heart_df is None:
    raise RuntimeError(
        "Could not load the heart disease dataset. "
        "Check the internet connection or dataset URL."
    )

heart_df = heart_df.loc[
    :,
    ~heart_df.columns.str.contains("^unnamed|^id$")
]

heart_df = heart_df.apply(
    pd.to_numeric, errors="coerce"
)

heart_df = heart_df.dropna(axis=1, how="all")

heart_features = [
    col for col in heart_df.columns
    if col != "target"
]

heart_df = heart_df.dropna(subset=["target"])

heart_df = heart_df.dropna(
    subset=heart_features,
    how="all"
)

heart_df["target"] = heart_df["target"].astype(int)

if heart_df["target"].nunique() != 2:
    raise ValueError(
        "The heart dataset must contain both target classes."
    )

print("Heart disease dataset loaded successfully.")
print("Dataset shape:", heart_df.shape)
print("Features:", heart_features)

display(heart_df.head())

Heart disease dataset loaded successfully.
Dataset shape: (303, 14)
Features: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [ ]:
diabetes_zero_columns = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

def prepare_features(data, zero_columns=None):

    data = data.copy()

    if zero_columns:
        for col in zero_columns:
            if col in data.columns:
                data[col] = data[col].replace(0, np.nan)

    return data

print("Data preparation function created.")

Data preparation function created.


In [ ]:
def train_disease_model(
    dataframe,
    feature_columns,
    target_column,
    zero_columns=None
):

    X = dataframe[feature_columns].copy()
    y = dataframe[target_column].astype(int)

    X = prepare_features(X, zero_columns)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    models = {
        "Logistic Regression": LogisticRegression(
            max_iter=2000
        ),

        "Random Forest": RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced"
        ),

        "Gradient Boosting": GradientBoostingClassifier(
            random_state=42
        )
    }

    results = []
    trained_models = {}

    for name, model in models.items():

        pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", model)
        ])

        pipeline.fit(X_train, y_train)

        predictions = pipeline.predict(X_test)

        probabilities = pipeline.predict_proba(
            X_test
        )[:, 1]

        auc = roc_auc_score(
            y_test, probabilities
        )

        results.append({
            "Model": name,
            "Accuracy": accuracy_score(
                y_test, predictions
            ),
            "Precision": precision_score(
                y_test, predictions, zero_division=0
            ),
            "Recall": recall_score(
                y_test, predictions, zero_division=0
            ),
            "F1 Score": f1_score(
                y_test, predictions, zero_division=0
            ),
            "ROC-AUC": auc
        })

        trained_models[name] = pipeline

    results_df = pd.DataFrame(results)

    best_model_name = results_df.sort_values(
        "ROC-AUC",
        ascending=False
    ).iloc[0]["Model"]

    return {
        "model": trained_models[best_model_name],
        "model_name": best_model_name,
        "features": feature_columns,
        "zero_columns": zero_columns or [],
        "results": results_df,
        "X_test": X_test,
        "y_test": y_test
    }


diabetes_bundle = train_disease_model(
    diabetes_df,
    diabetes_features,
    "Outcome",
    diabetes_zero_columns
)

heart_bundle = train_disease_model(
    heart_df,
    heart_features,
    "target"
)

print("Diabetes best model:", diabetes_bundle["model_name"])
print("Heart disease best model:", heart_bundle["model_name"])

print("\nDIABETES MODEL COMPARISON")
display(diabetes_bundle["results"].round(3))

print("\nHEART DISEASE MODEL COMPARISON")
display(heart_bundle["results"].round(3))

Diabetes best model: Gradient Boosting
Heart disease best model: Random Forest

DIABETES MODEL COMPARISON


,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.708,0.600,0.500,0.545,0.813
1,Random Forest,0.734,0.638,0.556,0.594,0.812
2,Gradient Boosting,0.760,0.689,0.574,0.626,0.831



HEART DISEASE MODEL COMPARISON


,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.803,0.769,0.909,0.833,0.869
1,Random Forest,0.803,0.756,0.939,0.838,0.903
2,Gradient Boosting,0.820,0.789,0.909,0.845,0.879


In [ ]:
def predict_risk(bundle, values):

    if len(values) != len(bundle["features"]):
        raise ValueError("Incorrect number of input features.")

    input_df = pd.DataFrame(
        [values],
        columns=bundle["features"]
    )

    input_df = prepare_features(
        input_df,
        bundle["zero_columns"]
    )

    probability = bundle["model"].predict_proba(
        input_df
    )[0][1]

    return float(probability)


def get_risk_category(probability):

    percentage = probability * 100

    if percentage < 30:
        return "Low Risk"

    elif percentage < 60:
        return "Moderate Risk"

    return "High Risk"


def generate_health_explanation(disease, probability):

    percentage = probability * 100
    category = get_risk_category(probability)

    if category == "Low Risk":
        recommendation = (
            "Maintain a balanced diet, exercise regularly, "
            "sleep adequately and attend routine health checkups."
        )

    elif category == "Moderate Risk":
        recommendation = (
            "Monitor relevant health indicators, maintain a "
            "balanced lifestyle and discuss your results "
            "with a healthcare professional."
        )

    else:
        recommendation = (
            "Consider seeking evaluation from a qualified "
            "healthcare professional. This prediction is not "
            "a medical diagnosis."
        )

    return f"""
DISEASE: {disease}

ESTIMATED MODEL RISK SCORE: {percentage:.2f}%

RISK CATEGORY: {category}

HEALTH RECOMMENDATION:
{recommendation}

IMPORTANT:
This result is generated by a machine-learning model
for educational and demonstration purposes only.
It is not a medical diagnosis.
"""


def create_gauge(probability):

    percentage = probability * 100

    fig = go.Figure(
        go.Indicator(
            mode="gauge+number",
            value=percentage,
            title={"text": "Estimated Risk Score"},
            number={"suffix": "%"},
            gauge={
                "axis": {"range": [0, 100]},
                "steps": [
                    {"range": [0, 30], "color": "#d1fae5"},
                    {"range": [30, 60], "color": "#fef3c7"},
                    {"range": [60, 100], "color": "#fee2e2"}
                ],
                "threshold": {
                    "line": {"color": "black", "width": 3},
                    "value": percentage
                }
            }
        )
    )

    fig.update_layout(height=300)

    return fig


print("Prediction functions are ready.")

Prediction functions are ready.


In [ ]:
def diabetes_prediction(
    pregnancies,
    glucose,
    blood_pressure,
    skin_thickness,
    insulin,
    bmi,
    pedigree,
    age
):

    values = [
        pregnancies,
        glucose,
        blood_pressure,
        skin_thickness,
        insulin,
        bmi,
        pedigree,
        age
    ]

    if any(value is None for value in values):
        return (
            "Please enter all diabetes input values.",
            "",
            None
        )

    try:
        values = [float(value) for value in values]

        probability = predict_risk(
            diabetes_bundle,
            values
        )

        category = get_risk_category(probability)

        result = (
            f"Diabetes Risk Category: {category}\n"
            f"Estimated Risk Score: {probability * 100:.2f}%\n"
            f"Selected Model: {diabetes_bundle['model_name']}"
        )

        explanation = generate_health_explanation(
            "Diabetes",
            probability
        )

        return (
            result,
            explanation,
            create_gauge(probability)
        )

    except Exception as error:
        return f"Prediction error: {error}", "", None


def heart_prediction(*values):

    if any(value is None for value in values):
        return (
            "Please enter all heart disease input values.",
            "",
            None
        )

    try:
        values = [float(value) for value in values]

        probability = predict_risk(
            heart_bundle,
            values
        )

        category = get_risk_category(probability)

        result = (
            f"Heart Disease Risk Category: {category}\n"
            f"Estimated Risk Score: {probability * 100:.2f}%\n"
            f"Selected Model: {heart_bundle['model_name']}"
        )

        explanation = generate_health_explanation(
            "Heart Disease",
            probability
        )

        return (
            result,
            explanation,
            create_gauge(probability)
        )

    except Exception as error:
        return f"Prediction error: {error}", "", None


print("Disease prediction callbacks are ready.")

Disease prediction callbacks are ready.


In [ ]:
def explore_data(dataset_name):

    if dataset_name == "Diabetes":
        data = diabetes_df
        bundle = diabetes_bundle

    else:
        data = heart_df
        bundle = heart_bundle

    description = data.describe().round(2)

    return (
        data.head(20),
        description,
        bundle["results"].round(3)
    )


print("Data Explorer function is ready.")

Data Explorer function is ready.


In [ ]:
def patient_monitoring(
    temperature,
    heart_rate,
    glucose,
    spo2
):

    readings = [
        temperature,
        heart_rate,
        glucose,
        spo2
    ]

    if any(value is None for value in readings):
        return (
            "Please enter all monitoring values.",
            pd.DataFrame(),
            None
        )

    alerts = []

    if temperature >= 38:
        alerts.append("High temperature detected.")

    if heart_rate > 100 or heart_rate < 60:
        alerts.append(
            "Heart rate is outside the illustrative range."
        )

    if glucose >= 200:
        alerts.append(
            "High glucose reading. Seek professional advice."
        )

    if spo2 < 95:
        alerts.append(
            "Oxygen saturation is below the illustrative threshold."
        )

    if not alerts:
        status = (
            "No threshold alerts from the entered readings."
        )

    else:
        status = (
            "Monitoring alerts:\n" +
            "\n".join("- " + alert for alert in alerts)
        )

    monitoring_data = pd.DataFrame({
        "Parameter": [
            "Temperature (°C)",
            "Heart Rate (bpm)",
            "Glucose (mg/dL)",
            "SpO2 (%)"
        ],
        "Reading": readings
    })

    fig = px.bar(
        monitoring_data,
        x="Parameter",
        y="Reading",
        title="Patient Monitoring Readings"
    )

    fig.update_layout(height=350)

    return status, monitoring_data, fig


print("Patient Monitoring function is ready.")

Patient Monitoring function is ready.


In [ ]:
def generate_pdf_report(
    patient_name,
    disease,
    risk_score,
    risk_category
):

    if not patient_name or not patient_name.strip():
        raise gr.Error("Please enter a patient name.")

    if risk_score is None or not 0 <= risk_score <= 100:
        raise gr.Error("Enter a risk score between 0 and 100.")

    if disease == "Diabetes":
        model_name = diabetes_bundle["model_name"]
    else:
        model_name = heart_bundle["model_name"]

    temp_file = tempfile.NamedTemporaryFile(
        delete=False,
        suffix=".pdf"
    )

    temp_file.close()

    pdf = canvas.Canvas(
        temp_file.name,
        pagesize=A4
    )

    width, height = A4
    y = height - 70

    pdf.setFont("Helvetica-Bold", 20)
    pdf.drawString(
        50,
        y,
        "Healthcare Risk Prediction Report"
    )

    y -= 50
    pdf.setFont("Helvetica", 12)

    report_lines = [
        f"Patient Name: {patient_name}",
        f"Disease Type: {disease}",
        f"Estimated Risk Score: {risk_score:.2f}%",
        f"Risk Category: {risk_category}",
        f"Machine Learning Model: {model_name}",
        "",
        "Recommendation:",
        "Please consult a qualified healthcare professional",
        "for interpretation and further evaluation.",
        "",
        "Disclaimer:",
        "This is an educational machine-learning report.",
        "It is not a medical diagnosis."
    ]

    for line in report_lines:
        pdf.drawString(50, y, line)
        y -= 25

    pdf.save()

    return temp_file.name


print("PDF report function is ready.")

PDF report function is ready.


In [ ]:
HEALTH_URLS = [
    "https://www.who.int/news-room/fact-sheets/detail/diabetes",
    "https://www.who.int/news-room/fact-sheets/detail/cardiovascular-diseases-(cvds)",
    "https://www.cdc.gov/diabetes/about/index.html",
    "https://www.cdc.gov/heart-disease/about/index.html"
]

def collect_health_information():

    documents = []

    headers = {
        "User-Agent": "Mozilla/5.0 HealthcareEducationProject/1.0"
    }

    for url in HEALTH_URLS:

        try:
            response = requests.get(
                url,
                headers=headers,
                timeout=20
            )

            response.raise_for_status()

            soup = BeautifulSoup(
                response.text,
                "html.parser"
            )

            for element in soup([
                "script",
                "style",
                "nav",
                "footer",
                "header"
            ]):
                element.decompose()

            paragraphs = [
                p.get_text(" ", strip=True)
                for p in soup.find_all(["p", "h1", "h2", "h3"])
            ]

            text = "\n".join(
                p for p in paragraphs if len(p) > 40
            )

            if len(text) > 300:
                documents.append({
                    "text": text,
                    "source": url
                })

                print("Collected:", url)

            else:
                print("Not enough readable content:", url)

        except Exception as error:
            print("Could not collect:", url, "|", str(error))

    if not documents:
        raise RuntimeError(
            "No healthcare pages could be downloaded. "
            "Check your internet connection and try again."
        )

    return documents


health_documents = collect_health_information()

print("\nTotal successfully collected pages:", len(health_documents))

Collected: https://www.who.int/news-room/fact-sheets/detail/diabetes
Collected: https://www.who.int/news-room/fact-sheets/detail/cardiovascular-diseases-(cvds)
Could not collect: https://www.cdc.gov/diabetes/about/index.html | 403 Client Error: Forbidden for url: https://www.cdc.gov/diabetes/about/index.html
Could not collect: https://www.cdc.gov/heart-disease/about/index.html | 403 Client Error: Forbidden for url: https://www.cdc.gov/heart-disease/about/index.html

Total successfully collected pages: 2


In [ ]:
def split_into_chunks(
    documents,
    chunk_size=700,
    overlap=100
):

    chunks = []

    for document in documents:

        words = document["text"].split()

        start = 0

        while start < len(words):

            end = start + chunk_size

            chunk_text = " ".join(
                words[start:end]
            )

            if len(chunk_text.strip()) > 50:
                chunks.append({
                    "text": chunk_text,
                    "source": document["source"]
                })

            start += chunk_size - overlap

    return chunks


health_chunks = split_into_chunks(
    health_documents
)

print("Total text chunks:", len(health_chunks))
print("\nSample chunk:")
print(health_chunks[0]["text"][:500])

Total text chunks: 6

Sample chunk:
Key facts The number of people living with diabetes rose from 200 million in 1990 to 830 million in 2022. Prevalence has been rising more rapidly in low- and middle-income countries than in high-income countries. More than half of people living with diabetes did not take medication for their diabetes in 2022. Diabetes treatment coverage was lowest in low- and middle-income countries. Diabetes causes blindness, kidney failure, heart attacks, stroke and lower limb amputation. In 2023, diabetes and


In [ ]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

chunk_texts = [
    chunk["text"]
    for chunk in health_chunks
]

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings = embeddings.astype("float32")

print("Embeddings generated.")
print("Embedding shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings generated.
Embedding shape: (6, 384)


In [ ]:
embedding_dimension = embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(
    embedding_dimension
)

faiss_index.add(embeddings)

print("FAISS vector database created.")
print("Total indexed chunks:", faiss_index.ntotal)

FAISS vector database created.
Total indexed chunks: 6


In [ ]:
def retrieve_health_information(
    question,
    top_k=4
):

    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    number_to_retrieve = min(
        top_k,
        len(health_chunks)
    )

    scores, indices = faiss_index.search(
        question_embedding,
        number_to_retrieve
    )

    retrieved = []

    for score, index in zip(
        scores[0],
        indices[0]
    ):

        if index == -1:
            continue

        retrieved.append({
            "text": health_chunks[index]["text"],
            "source": health_chunks[index]["source"],
            "similarity": float(score)
        })

    return retrieved


# Test retrieval
test_results = retrieve_health_information(
    "What are common risk factors for diabetes?"
)

for result in test_results:
    print("\nSource:", result["source"])
    print("Similarity:", round(result["similarity"], 3))
    print(result["text"][:300])


Source: https://www.who.int/news-room/fact-sheets/detail/diabetes
Similarity: 0.587
Key facts The number of people living with diabetes rose from 200 million in 1990 to 830 million in 2022. Prevalence has been rising more rapidly in low- and middle-income countries than in high-income countries. More than half of people living with diabetes did not take medication for their diabete

Source: https://www.who.int/news-room/fact-sheets/detail/cardiovascular-diseases-(cvds)
Similarity: 0.52
Cardiovascular diseases (CVDs) are a group of disorders of the heart and blood vessels. They include: Heart attacks and strokes are usually acute events and are mainly caused by a blockage that prevents blood from flowing to the heart or brain. The most common reason for this is a build-up of fatty 

Source: https://www.who.int/news-room/fact-sheets/detail/diabetes
Similarity: 0.52
to be achieved by 2030. To learn more about the Global Diabetes Compact, to access diabetes-related technical publications 

In [ ]:
def generate_llm_response(prompt):

    if API_PROVIDER in ["openai", "groq"]:

        response = llm_client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a healthcare education assistant. "
                        "Answer using the supplied retrieved context. "
                        "Do not invent medical facts or sources. "
                        "If the context is insufficient, say so. "
                        "Do not diagnose patients or prescribe treatment."
                    )
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.2
        )

        return response.choices[0].message.content

    elif API_PROVIDER == "gemini":

        response = llm_client.models.generate_content(
            model=LLM_MODEL,
            contents=prompt
        )

        return response.text

    elif API_PROVIDER == "anthropic":

        response = llm_client.messages.create(
            model=LLM_MODEL,
            max_tokens=1200,
            temperature=0.2,
            system=(
                "You are a healthcare education assistant. "
                "Use the supplied retrieved context. "
                "Do not invent medical facts or sources. "
                "Do not diagnose patients or prescribe treatment."
            ),
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        return "\n".join(
            block.text
            for block in response.content
            if hasattr(block, "text")
        )

    raise ValueError("Unsupported API provider.")


print("API response function created.")

API response function created.


In [ ]:
def rag_answer(question):

    if not question or not question.strip():
        return "Please enter a healthcare question.", ""

    try:

        retrieved = retrieve_health_information(
            question,
            top_k=4
        )

        if not retrieved:
            return (
                "No relevant information was found.",
                "No sources were retrieved."
            )

        context = ""

        for i, item in enumerate(retrieved, start=1):

            context += (
                f"\nSOURCE {i}: {item['source']}\n"
                f"CONTENT: {item['text']}\n"
            )

        prompt = f"""
USER QUESTION:
{question}

RETRIEVED HEALTHCARE INFORMATION:
{context}

INSTRUCTIONS:
1. Answer the user's question clearly.
2. Use the retrieved information as your evidence.
3. Do not invent facts.
4. If the retrieved content does not answer the question,
   explain that the available information is insufficient.
5. This is educational information, not a medical diagnosis.
6. Do not prescribe medicines or treatment.

Provide a clear and easy-to-understand answer.
"""

        answer = generate_llm_response(prompt)

        sources = "\n".join(
            f"{i}. {item['source']}"
            for i, item in enumerate(retrieved, start=1)
        )

        return answer, sources

    except Exception as error:

        return (
            "Unable to generate the RAG response. "
            "Check your API key, model access and internet connection.\n\n"
            f"Technical details: {str(error)}",
            ""
        )


print("RAG question-answering system is ready.")

RAG question-answering system is ready.


In [ ]:
test_question = "What are the common risk factors for diabetes?"

test_answer, test_sources = rag_answer(test_question)

print("RAG ANSWER")
print(test_answer)

print("\nRETRIEVED SOURCES")
print(test_sources)

RAG ANSWER
Unable to generate the RAG response. Check your API key, model access and internet connection.

Technical details: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-jOv67*************znkg. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

RETRIEVED SOURCES



In [ ]:
with gr.Blocks(
    theme=gr.themes.Soft(),
    title="AI-Powered Healthcare Risk Prediction"
) as demo:

    gr.Markdown(
        """
        # 🏥 AI-Powered Healthcare Risk Prediction

        ### Diabetes | Heart Disease | Health Explanation | Patient Monitoring

        An educational machine-learning healthcare application.
        """
    )

    # =========================================
    # TAB 1: DIABETES PREDICTION
    # =========================================

    with gr.Tab("Diabetes Prediction"):

        gr.Markdown("## Diabetes Risk Assessment")

        with gr.Row():

            with gr.Column():

                d_pregnancies = gr.Number(
                    label="Pregnancies",
                    value=1
                )

                d_glucose = gr.Number(
                    label="Glucose",
                    value=120
                )

                d_bp = gr.Number(
                    label="Blood Pressure",
                    value=70
                )

                d_skin = gr.Number(
                    label="Skin Thickness",
                    value=20
                )

            with gr.Column():

                d_insulin = gr.Number(
                    label="Insulin",
                    value=80
                )

                d_bmi = gr.Number(
                    label="BMI",
                    value=25
                )

                d_pedigree = gr.Number(
                    label="Diabetes Pedigree Function",
                    value=0.5
                )

                d_age = gr.Number(
                    label="Age",
                    value=30
                )

        diabetes_button = gr.Button(
            "Predict Diabetes Risk",
            variant="primary"
        )

        diabetes_result = gr.Textbox(
            label="Prediction Result",
            lines=3
        )

        diabetes_explanation = gr.Textbox(
            label="Health Risk Explanation",
            lines=8
        )

        diabetes_gauge = gr.Plot(
            label="Diabetes Risk Visualization"
        )

        diabetes_inputs = [
            d_pregnancies,
            d_glucose,
            d_bp,
            d_skin,
            d_insulin,
            d_bmi,
            d_pedigree,
            d_age
        ]

        diabetes_outputs = [
            diabetes_result,
            diabetes_explanation,
            diabetes_gauge
        ]

        diabetes_button.click(
            fn=diabetes_prediction,
            inputs=diabetes_inputs,
            outputs=diabetes_outputs
        )

        for component in diabetes_inputs:

            component.change(
                fn=diabetes_prediction,
                inputs=diabetes_inputs,
                outputs=diabetes_outputs
            )

    # =========================================
    # TAB 2: HEART DISEASE PREDICTION
    # =========================================

    with gr.Tab("Heart Disease Prediction"):

        gr.Markdown(
            "## Heart Disease Risk Assessment"
        )

        gr.Markdown(
            "Enter values corresponding to the heart dataset features."
        )

        heart_inputs = []

        with gr.Row():

            with gr.Column():

                for feature in heart_features[:7]:

                    component = gr.Number(
                        label=feature.replace("_", " ").title(),
                        value=None
                    )

                    heart_inputs.append(component)

            with gr.Column():

                for feature in heart_features[7:]:

                    component = gr.Number(
                        label=feature.replace("_", " ").title(),
                        value=None
                    )

                    heart_inputs.append(component)

        heart_button = gr.Button(
            "Predict Heart Disease Risk",
            variant="primary"
        )

        heart_result = gr.Textbox(
            label="Prediction Result",
            lines=3
        )

        heart_explanation = gr.Textbox(
            label="Health Risk Explanation",
            lines=8
        )

        heart_gauge = gr.Plot(
            label="Heart Disease Risk Visualization"
        )

        heart_outputs = [
            heart_result,
            heart_explanation,
            heart_gauge
        ]

        heart_button.click(
            fn=heart_prediction,
            inputs=heart_inputs,
            outputs=heart_outputs
        )

        for component in heart_inputs:

            component.change(
                fn=heart_prediction,
                inputs=heart_inputs,
                outputs=heart_outputs
            )

    # =========================================
    # TAB 3: DATA EXPLORER
    # =========================================

    with gr.Tab("Data Explorer"):

        gr.Markdown("## Dataset Analysis")

        dataset_choice = gr.Radio(
            choices=["Diabetes", "Heart Disease"],
            value="Diabetes",
            label="Select Dataset"
        )

        explore_button = gr.Button(
            "Explore Dataset",
            variant="primary"
        )

        dataset_preview = gr.Dataframe(
            label="Dataset Preview"
        )

        dataset_description = gr.Dataframe(
            label="Statistical Description"
        )

        model_comparison = gr.Dataframe(
            label="Machine Learning Model Comparison"
        )

        explore_button.click(
            fn=explore_data,
            inputs=dataset_choice,
            outputs=[
                dataset_preview,
                dataset_description,
                model_comparison
            ]
        )

    # =========================================
    # TAB 4: PATIENT MONITORING
    # =========================================

    with gr.Tab("Patient Monitoring"):

        gr.Markdown(
            """
            ## Patient Health Monitoring

            Enter sample readings to check for illustrative
            threshold alerts. This is not a live medical device.
            """
        )

        with gr.Row():

            temperature_input = gr.Number(
                label="Temperature (°C)",
                value=36.8
            )

            heart_rate_input = gr.Number(
                label="Heart Rate (bpm)",
                value=75
            )

        with gr.Row():

            glucose_input = gr.Number(
                label="Glucose (mg/dL)",
                value=110
            )

            spo2_input = gr.Number(
                label="SpO2 (%)",
                value=98
            )

        monitor_button = gr.Button(
            "Monitor Patient",
            variant="primary"
        )

        monitoring_status = gr.Textbox(
            label="Monitoring Status",
            lines=5
        )

        monitoring_table = gr.Dataframe(
            label="Patient Readings"
        )

        monitoring_chart = gr.Plot(
            label="Monitoring Visualization"
        )

        monitor_button.click(
            fn=patient_monitoring,
            inputs=[
                temperature_input,
                heart_rate_input,
                glucose_input,
                spo2_input
            ],
            outputs=[
                monitoring_status,
                monitoring_table,
                monitoring_chart
            ]
        )

    # =========================================
    # TAB 5: PDF REPORT
    # =========================================

    with gr.Tab("PDF Report"):

        gr.Markdown(
            "## Generate Healthcare Risk Report"
        )

        patient_name = gr.Textbox(
            label="Patient Name",
            placeholder="Enter patient name"
        )

        report_disease = gr.Dropdown(
            choices=["Diabetes", "Heart Disease"],
            value="Diabetes",
            label="Disease Type"
        )

        report_risk_score = gr.Number(
            label="Risk Score (%)",
            value=25
        )

        report_category = gr.Dropdown(
            choices=[
                "Low Risk",
                "Moderate Risk",
                "High Risk"
            ],
            value="Low Risk",
            label="Risk Category"
        )

        pdf_button = gr.Button(
            "Generate PDF Report",
            variant="primary"
        )

        pdf_output = gr.File(
            label="Download Healthcare Report"
        )

        pdf_button.click(
            fn=generate_pdf_report,
            inputs=[
                patient_name,
                report_disease,
                report_risk_score,
                report_category
            ],
            outputs=pdf_output
        )

    # =========================================
    # TAB 6: RAG HEALTHCARE ASSISTANT
    # =========================================

    with gr.Tab("RAG Healthcare Assistant"):

        gr.Markdown(
            """
            ## AI-Powered Healthcare Assistant

            Ask questions about diabetes, heart disease and
            general healthcare topics.

            The assistant retrieves relevant information from
            the indexed healthcare sources before generating
            its response.
            """
        )

        rag_question = gr.Textbox(
            label="Ask Your Healthcare Question",
            placeholder="Example: What are common risk factors for diabetes?",
            lines=3
        )

        rag_button = gr.Button(
            "Ask Healthcare Assistant",
            variant="primary"
        )

        rag_response = gr.Textbox(
            label="RAG Generated Answer",
            lines=12
        )

        rag_sources = gr.Textbox(
            label="Retrieved Information Sources",
            lines=5
        )

        rag_button.click(
            fn=rag_answer,
            inputs=rag_question,
            outputs=[
                rag_response,
                rag_sources
            ]
        )

    gr.Markdown(
        """
        ---

        **Disclaimer:** This application is an academic
        machine-learning prototype. Predictions and monitoring
        alerts are not a substitute for professional medical advice.
        """
    )

print("Healthcare website interface created successfully.")

Healthcare website interface created successfully.


In [ ]:
demo.queue()

demo.launch(
    share=True,
    debug=True,
    show_error=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://95df37e2149f81acb5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://95df37e2149f81acb5.gradio.live
